In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install python-mecab-ko

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.9/573.9 kB 15.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 51.8 MB/s eta 0:00:00


In [3]:
import re
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from mecab import MeCab
mecab = MeCab()
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer = Tokenizer()
from tensorflow.keras.preprocessing.sequence import pad_sequences
max_len = 75
import pickle

In [38]:
# 저장된 모델 로드
re_bst_loaded = lgb.Booster(model_file='/content/drive/MyDrive/2023데청캠/팀플/합불예측/LGBMmodel.txt')

#Tokenizer Object 파일로드
with open('/content/drive/MyDrive/2023데청캠/팀플/합불예측/tokenizer.pickle', 'rb') as handle:
    loaded_tokenizer = pickle.load(handle)

In [47]:
def predict_sen(new_sentence, model):
    new_sentence = re.sub(r'[^ㄱ-ㅎㅏ-ㅣ가-힣 ]','', new_sentence)
    new_sentence = mecab.morphs(new_sentence) # 토큰화
    encoded = loaded_tokenizer.texts_to_sequences([new_sentence]) # 정수 인코딩
    pad_new = pad_sequences(encoded, maxlen = max_len)

    pred_prob = float(model.predict(pad_new))
    print(pred_prob)

    # 결과 출력
    if pred_prob >= 0.5:
        result = f"This sentence is {pred_prob*100:.2f}% likely to be 합격."
    else:
        result = f"This sentence is {100-pred_prob*100:.2f}% likely to be 불합격."

    return result

In [48]:
input_sentence = """대답을 잘 못하였다."""
predicted_result = predict_sen(input_sentence, re_bst_loaded)
print(predicted_result)

0.12124101094471229
This sentence is 87.88% likely to be 불합격.


In [50]:
input_sentence = """질문 의도 파악이 어려웠다."""
predicted_result = predict_sen(input_sentence, re_bst_loaded)
print(predicted_result)

0.4658789095381718
This sentence is 53.41% likely to be 불합격.


In [51]:
input_sentence = """대답을 잘 하고 나왔다."""
predicted_result = predict_sen(input_sentence, re_bst_loaded)
print(predicted_result)

0.8822932971679587
This sentence is 88.23% likely to be 합격.


In [52]:
input_sentence = """평이하였음"""
predicted_result = predict_sen(input_sentence, re_bst_loaded)
print(predicted_result)

0.5784578304617201
This sentence is 57.85% likely to be 합격.


In [53]:
input_sentence = """합격이다."""
predicted_result = predict_sen(input_sentence, re_bst_loaded)
print(predicted_result)

0.7945107585071121
This sentence is 79.45% likely to be 합격.


In [54]:
input_sentence = """불합격이다."""
predicted_result = predict_sen(input_sentence, re_bst_loaded)
print(predicted_result)

0.26934888454632355
This sentence is 73.07% likely to be 불합격.
